# DataSphere smoke test: Qwen2-Audio, 5 pilot items

Repository: `/home/jupyter/project/slm-audio-evidence`  
Storage root: `/home/jupyter/filestore/space1/slm-audio-evidence`  
Upload: `<storage root>/input/pilot_audio.zip`

In [ ]:
!nvidia-smi -L

In [ ]:
# Persistent storage paths.
from pathlib import Path
import os

FILESTORE = Path("/home/jupyter/filestore/space1")
if not FILESTORE.is_dir():
    raise RuntimeError(
        f"Storage mount not found: {FILESTORE}"
    )

STORAGE_ROOT = FILESTORE / "slm-audio-evidence"
INPUT_DIR = STORAGE_ROOT / "input"
HF_HOME = STORAGE_ROOT / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
PYTHON_DEPS = STORAGE_ROOT / "python"
PIP_CACHE_DIR = STORAGE_ROOT / "pip_cache"
RUNS_DIR = STORAGE_ROOT / "results" / "datasphere_smoke"
PILOT_DATA = STORAGE_ROOT / "pilot_data"

for path in (INPUT_DIR, HF_HUB_CACHE, PYTHON_DEPS, PIP_CACHE_DIR, RUNS_DIR, PILOT_DATA):
    path.mkdir(parents=True, exist_ok=True)

# Used by shell commands below.
os.environ.update({
    "HF_HOME": str(HF_HOME),
    "HF_HUB_CACHE": str(HF_HUB_CACHE),
    "PYTHON_DEPS": str(PYTHON_DEPS),
    "PIP_CACHE_DIR": str(PIP_CACHE_DIR),
    "RUNS_DIR": str(RUNS_DIR),
    "PILOT_DATA": str(PILOT_DATA),
})

probe = STORAGE_ROOT / ".write_test"
probe.write_text("ok", encoding="utf-8")
probe.unlink()
print(f"Writable storage: {STORAGE_ROOT}")

In [ ]:
# Repository: /home/jupyter/project/slm-audio-evidence
%cd /home/jupyter/project
!test -d slm-audio-evidence/.git || git clone https://github.com/ladnlav/slm-audio-evidence.git
%cd slm-audio-evidence

In [ ]:
# Python packages: <storage>/slm-audio-evidence/python
from pathlib import Path
import os
import subprocess
import sys

deps = Path(os.environ["PYTHON_DEPS"])
marker = deps / ".qwen_deps_v1_ready"
packages = [
    "transformers==4.45.2",
    "tokenizers==0.20.3",
    "huggingface-hub==0.25.2",
    "safetensors==0.4.5",
    "accelerate==0.34.2",
    "bitsandbytes==0.43.3",
    "librosa==0.10.2.post1",
    "soundfile==0.12.1",
    "soxr==0.3.7",
    "lazy-loader==0.4",
]

if not marker.is_file():
    install_env = os.environ.copy()
    install_env.update({
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PYTHONNOUSERSITE": "1",
    })
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps", "--target", str(deps), *packages],
        check=True,
        env=install_env,
    )
    marker.touch()

deps_str = str(deps)
if deps_str not in sys.path:
    sys.path.insert(0, deps_str)

import torch
import torchaudio
import transformers

print(torch.__version__, torchaudio.__version__, transformers.__version__)

In [ ]:
# Audio: <storage>/slm-audio-evidence/pilot_data
from pathlib import Path
import os
import zipfile

archive = Path(os.environ["PILOT_DATA"]).parent / "input" / "pilot_audio.zip"
stored_audio = Path(os.environ["PILOT_DATA"]) / "data" / "audio" / "spoken_squad_test"
if not stored_audio.is_dir():
    if not archive.is_file():
        raise FileNotFoundError(
            f"Upload pilot_audio.zip to: {archive}"
        )
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(Path(os.environ["PILOT_DATA"]))

repo_audio = Path.cwd() / "data" / "audio" / "spoken_squad_test"
if not repo_audio.exists():
    repo_audio.symlink_to(stored_audio, target_is_directory=True)

wav_count = len(list(repo_audio.glob("*.wav")))
if wav_count < 5:
    raise RuntimeError(f"Only {wav_count} WAV files found in {repo_audio}")
print(f"Audio ready: {wav_count} files -> {stored_audio}")

In [ ]:
# Model cache: <storage>/slm-audio-evidence/huggingface
import os
import sys
from src.inference import main

os.environ.pop("TRANSFORMERS_CACHE", None)
previous_argv = sys.argv
sys.argv = [
    "src.inference",
    "--model", "qwen2audio",
    "--strategy", "plain",
    "--data", "data/manifests/pilot.jsonl",
    "--out", os.environ["RUNS_DIR"],
    "--limit", "5",
]
try:
    main()
finally:
    sys.argv = previous_argv

In [ ]:
# Results: <storage>/slm-audio-evidence/results/datasphere_smoke
from pathlib import Path
import os

response_files = sorted(Path(os.environ["RUNS_DIR"]).glob("qwen2audio_plain_*/responses.jsonl"))
if not response_files:
    raise FileNotFoundError("responses.jsonl was not created")
responses = response_files[-1]
completed = sum(1 for line in responses.read_text(encoding="utf-8").splitlines() if line.strip())
assert completed == 5, f"Expected 5 responses, got {completed}: {responses}"
print(f"A0 smoke test complete: {completed}/5 responses -> {responses}")